# Phase 2 · Notebook 1: Market Data Fundamentals
### OHLCV data · Returns · Log-returns

---

This notebook covers the absolute basics of financial price data.
By the end you will:
- Understand what OHLCV data is and where it comes from
- Know the difference between simple returns and log-returns
- Know *why* quants use log-returns (and when not to)
- Be able to fetch real BTC data from Binance's free API
- Visualise price and volume in a clean chart

> **Rule:** Run every cell. Change numbers. Break things. That's how this sticks.

---
## 1. What is OHLCV data?

Every price bar (also called a *candle*) summarises all trades that happened in a time window:

| Field | Meaning |
|---|---|
| **O**pen | Price of the very first trade in the window |
| **H**igh | Highest price any trade occurred at |
| **L**ow | Lowest price any trade occurred at |
| **C**lose | Price of the very last trade in the window |
| **V**olume | Total quantity traded during the window |

The window (called the *interval* or *timeframe*) can be 1 minute, 1 hour, 1 day — whatever you choose.

**Why close price?** Most analysis uses the close price because it's the "consensus" price at the end of the period — it reflects the final balance of buyers and sellers.

In [ ]:
# Install dependencies if needed
# !pip install requests pandas numpy matplotlib

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Clean plot style
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded ✓')

---
## 2. Fetching real data from Binance

Binance provides a completely free public API — no account or API key required.
We will fetch hourly BTC/USDT bars for the last ~40 days (1000 hours).

In [ ]:
def fetch_ohlcv(symbol='BTCUSDT', interval='1h', limit=1000):
    """
    Fetch OHLCV data from Binance public API.
    No API key required.
    
    symbol:   trading pair, e.g. 'BTCUSDT', 'ETHUSDT'
    interval: '1m', '5m', '15m', '1h', '4h', '1d'
    limit:    number of bars (max 1000)
    """
    url = 'https://api.binance.com/api/v3/klines'
    params = {'symbol': symbol, 'interval': interval, 'limit': limit}
    
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    raw = response.json()
    
    df = pd.DataFrame(raw, columns=[
        'timestamp', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_volume', 'trades',
        'taker_buy_base', 'taker_buy_quote', 'ignore'
    ])
    
    # Keep only what we need
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']].copy()
    
    # Convert types
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    
    df.set_index('timestamp', inplace=True)
    return df


# Fetch BTC hourly data
btc = fetch_ohlcv('BTCUSDT', '1h', 1000)

print(f'Fetched {len(btc)} bars')
print(f'From: {btc.index[0]}')
print(f'To:   {btc.index[-1]}')
print()
btc.head()

In [ ]:
# Basic statistics on the close price
# Get comfortable reading these — you'll see them constantly
btc['close'].describe()

In [ ]:
# --- Plot 1: Price and volume ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), 
                                gridspec_kw={'height_ratios': [3, 1]},
                                sharex=True)

# Price
ax1.plot(btc.index, btc['close'], color='steelblue', linewidth=1)
ax1.fill_between(btc.index, btc['close'], btc['close'].min(), 
                  alpha=0.08, color='steelblue')
ax1.set_ylabel('BTC Price (USDT)')
ax1.set_title('BTC/USDT — Hourly Close Price & Volume', fontsize=13)

# Volume
ax2.bar(btc.index, btc['volume'], color='steelblue', alpha=0.4, width=0.03)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')

fig.tight_layout()
plt.show()

print("Notice: volume spikes often coincide with large price moves.")
print("This is why volume is used as a signal confirmation filter.")

---
## 3. Simple returns vs. log-returns

This is one of the most important concepts in quant finance. Let's build the intuition from scratch.

### 3a. Simple return

The most natural way to measure change:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

If BTC goes from $100 to $110, simple return = 10%.
If it then goes from $110 back to $100, simple return = -9.09%.

### 3b. Log-return

$$r_t^{\log} = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

If BTC goes from $100 to $110, log-return = ln(110/100) = 9.53%.
If it then goes from $110 back to $100, log-return = ln(100/110) = -9.53%.

### Why quants use log-returns:

1. **Symmetry** — a +10% move and a -10% move are equal in magnitude (as above)
2. **Additivity** — multi-period return = sum of single-period log-returns  
   (simple returns multiply, which is messier)
3. **Statistical properties** — log-returns are closer to normally distributed,  
   which makes most statistical tests valid
4. **Numerical stability** — for very long time series or extreme prices

### When NOT to use log-returns:
- Position sizing (you need dollar P&L, not logs)
- Portfolio aggregation (you can't add log-returns across assets)
- Communicating results to non-quants

In [ ]:
# Calculate both return types
btc['simple_return'] = btc['close'].pct_change()          # (P_t - P_{t-1}) / P_{t-1}
btc['log_return']    = np.log(btc['close'] / btc['close'].shift(1))  # ln(P_t / P_{t-1})

# Drop the first row (NaN — no previous price to compare to)
returns = btc[['simple_return', 'log_return']].dropna()

print('Simple return stats:')
print(returns['simple_return'].describe().round(6))
print()
print('Log return stats:')
print(returns['log_return'].describe().round(6))

In [ ]:
# --- Plot 2: Comparing return distributions ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, label, color in [
    (axes[0], 'simple_return', 'Simple returns', 'steelblue'),
    (axes[1], 'log_return',    'Log-returns',    'darkorange')
]:
    returns[col].hist(bins=80, ax=ax, color=color, alpha=0.7, edgecolor='none')
    ax.axvline(returns[col].mean(), color='red', linestyle='--', linewidth=1.5, label='mean')
    ax.set_title(label)
    ax.set_xlabel('Return')
    ax.set_ylabel('Frequency')
    ax.legend()

fig.suptitle('Return distributions — notice they look nearly identical for hourly data', 
             fontsize=11, y=1.02)
fig.tight_layout()
plt.show()

print("At short timeframes (hourly), log-returns ≈ simple returns.")
print("The difference matters more over longer horizons or larger price moves.")

In [ ]:
# --- The additivity property demonstrated ---
# Log-returns are additive: sum of daily log-returns = total log-return over the period

n_bars = 24  # last 24 hours

# Method 1: sum of log-returns
sum_of_logs = btc['log_return'].tail(n_bars).sum()

# Method 2: single log-return over the whole period
start_price = btc['close'].iloc[-(n_bars+1)]
end_price   = btc['close'].iloc[-1]
total_log   = np.log(end_price / start_price)

print(f'Sum of {n_bars} hourly log-returns:  {sum_of_logs:.6f}')
print(f'Single log-return over same period: {total_log:.6f}')
print(f'Difference: {abs(sum_of_logs - total_log):.10f}  ← effectively zero')
print()
print('This would NOT work with simple returns (they multiply, not add).')

---
## 4. Rolling statistics — your first indicator building block

Almost every indicator is just a rolling calculation on price or returns.
Let's compute rolling mean and rolling standard deviation — the two most important.

**Rolling volatility** (rolling std of log-returns) is especially important:
it tells you how "turbulent" the market has been recently.

In [ ]:
# Rolling statistics
window = 24  # 24-hour rolling window

btc['rolling_mean']   = btc['close'].rolling(window).mean()
btc['rolling_std']    = btc['close'].rolling(window).std()
btc['rolling_vol']    = btc['log_return'].rolling(window).std()  # volatility

# Annualised volatility (multiply by sqrt of periods per year)
# Hourly data: 24 * 365 = 8760 hours per year
btc['annualised_vol'] = btc['rolling_vol'] * np.sqrt(24 * 365)

print(f'Current 24h rolling volatility (annualised): {btc["annualised_vol"].iloc[-1]:.1%}')
print(f'Average over full period:                    {btc["annualised_vol"].mean():.1%}')
print()
print('For context: S&P 500 annual vol ≈ 15-20%. BTC is much higher.')

In [ ]:
# --- Plot 3: Price with rolling mean, and volatility below ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7),
                                gridspec_kw={'height_ratios': [3, 1]},
                                sharex=True)

# Price + rolling mean
ax1.plot(btc.index, btc['close'],        color='steelblue',   linewidth=1,   label='Close price', alpha=0.8)
ax1.plot(btc.index, btc['rolling_mean'], color='darkorange',  linewidth=1.5, label=f'{window}h rolling mean')
ax1.set_ylabel('BTC Price (USDT)')
ax1.set_title(f'BTC/USDT — Price, Rolling Mean & Rolling Volatility (window={window}h)', fontsize=13)
ax1.legend()

# Annualised volatility
ax2.plot(btc.index, btc['annualised_vol'], color='crimson', linewidth=1)
ax2.fill_between(btc.index, btc['annualised_vol'], 0, alpha=0.15, color='crimson')
ax2.set_ylabel('Ann. Vol')
ax2.set_xlabel('Date')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

fig.tight_layout()
plt.show()

print('Notice how volatility clusters — high-vol periods tend to be followed by more high-vol.')
print('This is called volatility clustering, and it is a key stylised fact of financial markets.')

---
## 5. Exercises — do these before moving on

These will cement the concepts. Each one takes ~5 minutes.

**Exercise 1:** Fetch ETH/USDT data and plot it alongside BTC. Do they move together?

**Exercise 2:** Calculate the *correlation* between BTC and ETH log-returns.  
Hint: `returns_a.corr(returns_b)`

**Exercise 3:** Change the rolling window from 24 to 6 and then to 168 (1 week).  
What happens to the rolling mean line? Why?

**Exercise 4:** What was the single largest hourly log-return in your dataset?  
What date did it happen? What was the price before and after?

**Exercise 5 (harder):** Reconstruct the price series from log-returns.  
Start from `btc['close'].iloc[0]` and use `np.exp(log_returns).cumprod()`.  
Plot it against the original price — they should match exactly.

In [ ]:
# Your workspace for exercises

# Exercise 1: fetch ETH and compare
# eth = fetch_ohlcv('ETHUSDT', '1h', 1000)
# ...

# Exercise 4: largest single-hour move
# ...


---
## Summary

| Concept | Key takeaway |
|---|---|
| OHLCV | One row = one time window of price action. Close is most used. |
| Simple return | Intuitive. Use for P&L and communicating results. |
| Log-return | Additive, symmetric, better stats. Use for analysis. |
| Rolling mean | Smooths price — foundation of all moving average strategies. |
| Rolling vol | Measures recent turbulence. Spikes near big price moves. |
| Vol clustering | High-vol periods cluster. A key fact about real markets. |

---
**Next notebook:** Moving averages in depth — SMA, EMA, how they differ, and why momentum strategies use them.